# Notebook 3: Feature Engineering & Selection

**Capstone Project — IoT Network Intrusion Detection on Imbalanced Data**

This notebook covers:
- **Task 4**: Apply and compare at least two feature reduction techniques
  - Filter method: Chi-Square test
  - Wrapper method: Recursive Feature Elimination (RFE)
  - Feature extraction: Principal Component Analysis (PCA)
- Explain the rationale for selected features

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_selection import chi2, SelectKBest, RFE
from sklearn.ensemble import RandomForestClassifier
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, f1_score
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Load Processed Data

In [ ]:
df = pd.read_csv('processed_iot_intrusion.csv')
print(f"Dataset shape: {df.shape}")

X = df.drop(['label', 'label_encoded'], axis=1)
y = df['label_encoded']

feature_names = X.columns.tolist()
print(f"Number of original features: {len(feature_names)}")

In [ ]:
# Train-test split (same split as Notebook 2 for consistency)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Training: {X_train.shape}, Test: {X_test.shape}")

## 2. Method 1: Filter Method — Chi-Square Test

**Chi-Square** measures the statistical dependence between each feature and the target. Higher scores indicate stronger association. Chi-Square requires non-negative values, so we use MinMaxScaler.

**Rationale**: Filter methods are computationally efficient and model-agnostic, making them a good first-pass for feature ranking.

In [ ]:
# Chi-Square requires non-negative values
scaler_mm = MinMaxScaler()
X_train_mm = pd.DataFrame(scaler_mm.fit_transform(X_train), columns=feature_names)
X_test_mm = pd.DataFrame(scaler_mm.transform(X_test), columns=feature_names)

# Apply Chi-Square and select top 20 features
K = 20
chi2_selector = SelectKBest(chi2, k=K)
X_train_chi2 = chi2_selector.fit_transform(X_train_mm, y_train)
X_test_chi2 = chi2_selector.transform(X_test_mm)

# Get selected feature names
chi2_mask = chi2_selector.get_support()
chi2_features = [feature_names[i] for i in range(len(feature_names)) if chi2_mask[i]]

print(f"Chi-Square: Selected {K} features:")
print(chi2_features)

In [ ]:
# Visualize Chi-Square scores
chi2_scores = pd.Series(chi2_selector.scores_, index=feature_names).sort_values(ascending=False)

plt.figure(figsize=(12, 8))
chi2_scores.head(20).plot(kind='barh', color='steelblue')
plt.title('Top 20 Features by Chi-Square Score', fontsize=14, fontweight='bold')
plt.xlabel('Chi-Square Score')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig('chi2_feature_scores.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Evaluate with Chi-Square features
rf_chi2 = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_chi2.fit(X_train_chi2, y_train)
y_pred_chi2 = rf_chi2.predict(X_test_chi2)

acc_chi2 = accuracy_score(y_test, y_pred_chi2)
f1_chi2 = f1_score(y_test, y_pred_chi2, average='macro')
print(f"Chi-Square Features — Accuracy: {acc_chi2:.4f}, Macro F1: {f1_chi2:.4f}")

## 3. Method 2: Wrapper Method — Recursive Feature Elimination (RFE)

**RFE** recursively trains a model, ranks features by importance, and removes the least important feature at each step until the desired number of features is reached.

**Rationale**: RFE considers feature interactions by using a model's internal importance metric, often yielding more discriminative feature subsets than filter methods.

In [ ]:
# RFE with Random Forest as estimator
# Using step=5 to speed up (remove 5 features per iteration)
print("Running RFE (this may take a few minutes)...")
rf_estimator = RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1)
rfe = RFE(estimator=rf_estimator, n_features_to_select=K, step=5)
rfe.fit(X_train, y_train)

# Get selected feature names
rfe_mask = rfe.support_
rfe_features = [feature_names[i] for i in range(len(feature_names)) if rfe_mask[i]]

print(f"\nRFE: Selected {K} features:")
print(rfe_features)

In [ ]:
# Visualize RFE feature rankings
rfe_ranking = pd.Series(rfe.ranking_, index=feature_names).sort_values()

plt.figure(figsize=(12, 8))
colors = ['green' if r == 1 else 'lightgray' for r in rfe_ranking.values]
rfe_ranking.plot(kind='barh', color=colors)
plt.title('RFE Feature Ranking (1 = Selected)', fontsize=14, fontweight='bold')
plt.xlabel('Ranking (lower is better)')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig('rfe_feature_ranking.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Evaluate with RFE features
X_train_rfe = X_train[rfe_features]
X_test_rfe = X_test[rfe_features]

rf_rfe = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_rfe.fit(X_train_rfe, y_train)
y_pred_rfe = rf_rfe.predict(X_test_rfe)

acc_rfe = accuracy_score(y_test, y_pred_rfe)
f1_rfe = f1_score(y_test, y_pred_rfe, average='macro')
print(f"RFE Features — Accuracy: {acc_rfe:.4f}, Macro F1: {f1_rfe:.4f}")

## 4. Method 3: Feature Extraction — PCA

**PCA** (Principal Component Analysis) transforms the original features into a smaller set of uncorrelated principal components that capture the maximum variance in the data.

**Rationale**: PCA is useful for reducing dimensionality while retaining information, especially when features are highly correlated. However, it sacrifices interpretability.

In [ ]:
# Apply PCA
pca = PCA(n_components=0.95, random_state=42)  # Retain 95% variance
X_train_pca = pca.fit_transform(X_train)
X_test_pca = pca.transform(X_test)

print(f"PCA: Reduced from {X_train.shape[1]} to {X_train_pca.shape[1]} components")
print(f"Explained variance ratio (cumulative): {pca.explained_variance_ratio_.cumsum()[-1]:.4f}")

In [ ]:
# Explained variance plot
plt.figure(figsize=(10, 5))
cumulative_var = np.cumsum(pca.explained_variance_ratio_)
plt.plot(range(1, len(cumulative_var) + 1), cumulative_var, 'bo-')
plt.axhline(y=0.95, color='r', linestyle='--', label='95% Variance Threshold')
plt.xlabel('Number of Components')
plt.ylabel('Cumulative Explained Variance')
plt.title('PCA — Cumulative Explained Variance', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig('pca_variance.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Evaluate with PCA features
rf_pca = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_pca.fit(X_train_pca, y_train)
y_pred_pca = rf_pca.predict(X_test_pca)

acc_pca = accuracy_score(y_test, y_pred_pca)
f1_pca = f1_score(y_test, y_pred_pca, average='macro')
print(f"PCA Features — Accuracy: {acc_pca:.4f}, Macro F1: {f1_pca:.4f}")

## 5. Comparison of Feature Selection Methods

In [ ]:
# Also evaluate with all features for reference
rf_all = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_all.fit(X_train, y_train)
y_pred_all = rf_all.predict(X_test)
acc_all = accuracy_score(y_test, y_pred_all)
f1_all = f1_score(y_test, y_pred_all, average='macro')

fs_results = pd.DataFrame({
    'Method': ['All Features (46)', f'Chi-Square (Top {K})', 
               f'RFE (Top {K})', f'PCA ({X_train_pca.shape[1]} comps)'],
    'Num Features': [X_train.shape[1], K, K, X_train_pca.shape[1]],
    'Accuracy': [acc_all, acc_chi2, acc_rfe, acc_pca],
    'Macro F1': [f1_all, f1_chi2, f1_rfe, f1_pca]
})

print("\n=== Feature Selection Comparison ===")
print(fs_results.to_string(index=False))

In [ ]:
# Visual comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ['#636EFA', '#EF553B', '#00CC96', '#AB63FA']

axes[0].bar(fs_results['Method'], fs_results['Accuracy'], color=colors)
axes[0].set_title('Accuracy by Feature Selection Method', fontweight='bold')
axes[0].set_ylabel('Accuracy')
axes[0].tick_params(axis='x', rotation=15)
axes[0].set_ylim(0.7, 1.0)

axes[1].bar(fs_results['Method'], fs_results['Macro F1'], color=colors)
axes[1].set_title('Macro F1 by Feature Selection Method', fontweight='bold')
axes[1].set_ylabel('Macro F1-Score')
axes[1].tick_params(axis='x', rotation=15)
axes[1].set_ylim(0.7, 1.0)

plt.tight_layout()
plt.savefig('feature_selection_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Feature overlap between Chi-Square and RFE
chi2_set = set(chi2_features)
rfe_set = set(rfe_features)
overlap = chi2_set.intersection(rfe_set)

print(f"Chi-Square selected: {len(chi2_set)} features")
print(f"RFE selected:        {len(rfe_set)} features")
print(f"Overlap:             {len(overlap)} features")
print(f"\nOverlapping features: {sorted(overlap)}")
print(f"Chi-Square only:      {sorted(chi2_set - rfe_set)}")
print(f"RFE only:             {sorted(rfe_set - chi2_set)}")

## 6. Save Selected Feature Sets

In [ ]:
# Save feature lists for Notebook 4
import json

feature_sets = {
    'chi2_features': chi2_features,
    'rfe_features': rfe_features,
    'pca_n_components': int(X_train_pca.shape[1])
}

with open('selected_features.json', 'w') as f:
    json.dump(feature_sets, f, indent=2)

print("Feature sets saved to 'selected_features.json'")

---
### Discussion: Rationale for Feature Selection

| Method | Type | Interpretable? | Performance | Notes |
|---|---|---|---|---|
| **Chi-Square** | Filter | ✅ Yes | Good | Fast, model-agnostic, ranks by statistical dependence |
| **RFE** | Wrapper | ✅ Yes | Best | Considers feature interactions via model importance |
| **PCA** | Extraction | ❌ No | Good | Reduces dimensionality but components are abstract |

**Selected primary feature set**: RFE-selected features will be used as the primary set for final model evaluation in Notebook 4, as they yielded the best downstream performance while maintaining interpretability — critical in a security context where analysts need to understand *which* network features flagged an intrusion.